# Blood gene signatures for active vs latent TB in HIV− and HIV+ African adults

**Computing Bioinformatics group project** · Matthew Nguithi (162156) · Ray Lukorito (168110) · William Githinji (110592)

<a href="https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/tb-hiv-signatures/blob/main/notebooks/TB_HIV_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Research question
> Can blood gene expression distinguish **active tuberculosis** from **latent TB infection** in African adults, and does **HIV co-infection weaken** that signal?

**Hypothesis.** Active TB shows strong up-regulation of interferon-driven immune genes in both HIV groups, but the active-vs-latent difference (and the accuracy of classifiers built on it) is lower in HIV-positive patients.

### Objectives
1. Identify genes differentially expressed between active and latent TB, separately for HIV− and HIV+ patients (FDR-corrected), and compare how much the two lists overlap.
2. Interpret the top genes with GO/pathway enrichment and a STRING protein-interaction network analysed in NetworkX to find hub genes.
3. Build and evaluate ML classifiers for active vs latent TB against proper baselines, and test whether a model trained on HIV− patients still works on HIV+ patients.

### Data
* **GSE37250** (Kaforou et al., 2013, *PLoS Med*): whole-blood Illumina HumanHT-12 V4 arrays, adults from Cape Town, South Africa (n = 300) and Karonga, Malawi (n = 237), labelled active TB / latent TB / other disease and HIV+ / HIV−.
* **GSE19491** (Berry et al., 2010, *Nature*): held back for external validation. Its TB samples are in sub-series GSE19439, GSE19444 (London) and GSE19442 (South Africa). **HIV− only**, so it can check the HIV− result but not the HIV+ one.

### Notebook stages
| # | Stage | Section |
|---|---|---|
| 1 | Load data and labels, keep active + latent TB | §1 |
| 2 | Quality control | §2 |
| 3 | PCA / exploratory analysis and batch check | §2 |
| 4 | Differential expression (limma-style, BH FDR), overlap, TB × HIV interaction | §3 |
| 5 | GO / pathway enrichment (Enrichr) | §4 |
| 6 | STRING network + NetworkX hub genes | §5 |
| 7 | ML with baselines, HIV− → HIV+ transfer, site transfer | §6 |
| 8 | External validation on GSE19491 | §7 |

All analysis code lives in the `tbhiv` package (`src/tbhiv/`, unit-tested in `tests/`); this notebook runs it stage by stage and explains each result. Runtime on a standard Colab CPU: roughly 10–20 minutes, most of it spent downloading data and cross-validating.

## 0. Setup

In [ ]:
# Colab: clone the project repository (set REPO_URL once the group repo is on GitHub).
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/tb-hiv-signatures"

import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not pathlib.Path("src/tbhiv").exists():
    if not pathlib.Path("tb-hiv-signatures").exists():
        !git clone -q {REPO_URL}
    %cd tb-hiv-signatures
    %pip install -q -r requirements.txt

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "tbhiv").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import tbhiv
from tbhiv import Config, pipeline, de, ml, signatures
from tbhiv.report import Report

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.precision", 3, "display.max_columns", 30, "display.width", 160)
%matplotlib inline
print("tbhiv", tbhiv.__version__)

### Configuration
Every tunable setting (FDR cut-off, fold-change threshold, CV folds/repeats, number of genes kept inside each fold, target sensitivity, etc.) is in `Config`; see `src/tbhiv/config.py`.

**If NCBI is slow or unreachable**, download `GSE37250_series_matrix.txt.gz` (and the GSE19439/42/44 matrices) from GEO, upload them (or put them on Google Drive), and fill in `local_matrix_files`.

`DEMO_SYNTHETIC` swaps in **synthetic** GEO-format files so the code can be tested offline. It must stay `False` for the real analysis; anything produced in demo mode is not a result.

In [ ]:
DEMO_SYNTHETIC = os.environ.get("TBHIV_DEMO", "0") == "1"   # leave False for the real analysis

cfg = Config(
    data_dir="data",
    results_dir="results",
    cv_repeats=5,          # 5 x 5-fold CV; lower to 2 for a quicker run
    # local_matrix_files={"GSE37250": "/content/drive/MyDrive/GSE37250_series_matrix.txt.gz"},
)

DATA_NOTE = ""
if DEMO_SYNTHETIC:
    from tbhiv.synthetic import make_synthetic_geo
    syn = make_synthetic_geo("data/synthetic", seed=1)
    cfg.local_matrix_files, cfg.local_platform_files = syn["local_matrix_files"], syn["local_platform_files"]
    cfg.gmt_files, cfg.online = syn["gmt_files"], False
    cfg.results_dir = pathlib.Path("results_SYNTHETIC_DEMO")
    DATA_NOTE = "SYNTHETIC DEMO DATA - these numbers are NOT research results."
    display(Markdown(f"<span style='color:#d03b3b;font-weight:bold'>⚠ {DATA_NOTE}</span>"))

cfg.ensure_dirs()
rep = Report(cfg.results_dir, "TB vs latent TB in HIV- and HIV+ African adults: results summary")
cfg

## 1. Load data and labels
The loader downloads each GEO series matrix, parses the sample characteristics into tidy labels (`tb_status`, `hiv_status`, `site`), log2-transforms if needed, quantile-normalises, and maps Illumina probes to gene symbols (keeping the highest-expressed probe per gene). Platform annotation (GPL10558 / GPL6947) is fetched with **GEOparse**.

In [ ]:
loaded = pipeline.stage_load(cfg, rep)
main, val = loaded["main"], loaded["val"]
display(loaded["logs"])
display(loaded["counts"])

We keep **active TB (ATB)** and **latent TB (LTBI)** only, and record the counts before and after (the *other disease* group is outside our question).

In [ ]:
sel = pipeline.stage_select(main, cfg, rep)
print("Before:", sel["selection"]["before"])
sel["counts"]

## 2. Quality control and exploratory analysis
* **Low-expression filter.** Roughly a third of Illumina probes measure genes that are not expressed in blood; dropping the bottom 30% by mean expression removes noise and reduces the multiple-testing burden.
* **Sample QC.** Each array's average correlation with all others; a robust z below −4 flags a likely failed array.
* **PCA** on the 5,000 most variable genes, coloured by TB status, HIV status and site. The table gives the share of each PC's variance explained by each factor (R²). A large R² for `site` means a Cape Town vs Karonga batch/site effect, so every differential-expression model below adjusts for site.

In [ ]:
q = pipeline.stage_qc(sel["ds"], cfg, rep)
ds = q["ds"]
print(q["filter"])
print("Arrays flagged as outliers:", int(q["qc_table"]["outlier"].sum()))
display(q["qc_table"][q["qc_table"]["outlier"]])
q["assoc"].style.format("{:.3f}").background_gradient(cmap="Blues", vmin=0, vmax=0.5)

> **Interpretation prompt.** Which factor dominates PC1 and PC2? If `site` explains much of an early PC, mention it as a batch effect in the presentation and point to the site covariate in the DE model. Flagged arrays are *reported*, not removed automatically; if one is extreme on the PCA plot, remove it explicitly here and say so in the write-up.

## 3. Differential expression (Objective 1)
For each HIV group separately we fit, for every gene:

$$\text{expression} = \beta_0 + \beta_{\text{TB}}\cdot[\text{active TB}] + \beta_{\text{site}}\cdot[\text{Malawi}] + \varepsilon$$

and test $\beta_{\text{TB}}$ (= log2 fold change, active vs latent) with a **moderated t-statistic**. As in *limma*, gene variances are shrunk towards a common prior estimated from all genes (empirical Bayes), which is more stable than a plain t-test at n ≈ 90 per group. P-values are corrected with **Benjamini–Hochberg**. A gene is significant at FDR < 0.05 and |log2FC| ≥ 0.5.

In [ ]:
d = pipeline.stage_de(ds, cfg, rep)
display(d["summary"])
cmp = d["compare"]
print(f"Shared significant genes: {cmp['n_both']}  |  Jaccard {cmp['jaccard']:.2f}  |  "
      f"hypergeometric p = {cmp['hypergeom_p']:.2e}  |  same direction: {cmp['shared_same_direction']}/{cmp['n_both']}")
print(f"Spearman correlation of log2FC across all genes (HIV- vs HIV+): {cmp['lfc_spearman_all_genes']:.2f}")
print(f"Empirical-Bayes prior d.f. (HIV-): {d['info']['HIV-']['d0']:.1f}")

In [ ]:
cols = ["log2FC", "AveExpr", "t", "adj.P.Val"]
print("Top 15 genes, HIV-"); display(d["res_neg"].head(15)[cols])
print("Top 15 genes, HIV+"); display(d["res_pos"].head(15)[cols])

How do the genes of the two published signatures behave in each HIV group?

In [ ]:
d["signature_genes"].round(3)

### Does HIV weaken the signal? (testing the hypothesis directly)
Comparing two gene lists only gives part of the answer, because list sizes depend on sample size and thresholds. We use two more direct tests:

1. **TB × HIV interaction model** on all active and latent TB patients: `expr ~ TB + HIV + TB:HIV + site`. The interaction term is the *change* in the active-vs-latent effect in HIV+ patients. A gene is *attenuated* if the interaction is significant (FDR < 0.05) and opposite in sign to the HIV− effect.
2. **Attenuation ratio**: across genes significant in the *pooled* analysis, the average signed effect in HIV+ divided by that in HIV− (1 = equally strong; below 1 = weaker in HIV+), with a bootstrap 95% CI that resamples patients. The genes are chosen from the pooled analysis on purpose: choosing them in HIV− would inflate the HIV− effects (the *winner's curse*) and bias the ratio below 1 even if HIV made no difference.

In [ ]:
att = d["attenuation"]
print(f"Attenuation ratio: {att['ratio']:.2f} (95% CI {att['ci_low']:.2f}-{att['ci_high']:.2f}) over {att['n_genes']} genes")
inter = d["interaction"]
print(f"Interaction test: {int(inter['attenuated'].sum())} attenuated, {int(inter['amplified'].sum())} amplified genes (FDR < {cfg.fdr})")
inter.head(20)[["tb_effect_HIVneg", "tb_effect_HIVpos", "interaction", "adj.P.Val", "attenuated"]]

## 4. Enrichment analysis (Objective 2)
Gene lists are sent to **Enrichr** (via `gseapy`) for GO Biological Process 2023, KEGG 2021 and Reactome 2022, with **all tested genes as the background** (otherwise any blood-expressed gene list looks enriched for immune terms). Lists analysed: up in active TB (HIV−), down in active TB (HIV−), up in active TB (HIV+), and genes significant **only in HIV−** (the signal HIV may erase).

In [ ]:
en = pipeline.stage_enrichment(d, ds, cfg, rep)
for name, r in en["results"].items():
    print(f"\n{name}: {len(en['lists'][name])} genes" + (f" - {r.attrs['note']}" if r.attrs.get("note") else ""))
    if len(r):
        display(r.head(8)[["library", "Term", "Overlap", "Adjusted P-value"]])

> **Interpretation prompt.** Do interferon-γ / type-I interferon signalling, antigen presentation or neutrophil degranulation terms dominate the active-TB list, as the hypothesis predicts? Which processes appear in the "HIV− only" list?

## 5. Protein-interaction network and hub genes (Objective 2)
The top 150 significant genes from each HIV group are sent to the **STRING** API (high confidence, score ≥ 700). Interactions are loaded into **NetworkX**; hubs are ranked by degree, then betweenness centrality, and communities are found by greedy modularity. If STRING is unreachable the code falls back to a co-expression network and says so.

In [ ]:
nw = pipeline.stage_network(d, ds, cfg, rep)
for g in ("HIV-", "HIV+"):
    print(g, nw[g]["summary"])
    display(nw[g]["hubs"].head(10).round(3))

## 6. Machine learning (Objective 3)
**Models.** Majority class (AUC 0.5 by construction); the published **Sweeney 3-gene score** (GBP5 + DUSP3 − KLF2) and **Kaforou 27-gene score**; **L1-penalised logistic regression** (penalty tuned by inner 3-fold CV); and a **random forest**.

**Evaluation.** 5-fold cross-validation repeated 5 times, stratified by TB status and site. The top-500-gene filter is fitted **inside each training fold**, so test patients never influence gene choice. We report AUC with bootstrap 95% CIs, plus sensitivity and specificity at the cut-off that gives ≥ 90% sensitivity (the WHO target for a TB triage test). Following our ethics safeguard, **misses are reported separately for HIV+ patients** rather than hidden in one overall score.

**Tests.** (a) CV within each HIV group; (b) one pooled model, with metrics broken down by HIV status and site at a single shared threshold; (c) **train on HIV−, test on HIV+** (and the reverse); (d) train on one site, test on the other.

In [ ]:
m = pipeline.stage_ml(ds, cfg, rep)
m["summary"][["group", "AUC", "AUC_low", "AUC_high", "sensitivity", "specificity", "missed_TB", "n_TB"]]

In [ ]:
print("Pooled model - one threshold, results split by subgroup")
display(m["subgroup"][["model", "by", "group", "n", "AUC", "sensitivity", "specificity", "missed_TB", "n_TB"]])
print("Train HIV-  ->  test HIV+")
display(m["transfer_neg_to_pos"][["AUC", "AUC_low", "AUC_high", "sensitivity", "missed_TB", "n_TB", "specificity", "specificity_group_threshold"]])
print("Train HIV+  ->  test HIV-")
display(m["transfer_pos_to_neg"][["AUC", "AUC_low", "AUC_high", "sensitivity", "missed_TB", "n_TB", "specificity", "specificity_group_threshold"]])
print("Leave one site out")
display(m["leave_site_out"][["train_site", "test_site", "AUC", "AUC_low", "AUC_high"]])

Which genes do the final HIV− models rely on, and how many are in the published 27-gene signature or among the network hubs?

In [ ]:
feats = m["features"]
k27 = set(signatures.signature_genes("Kaforou_27"))
hubs = set(nw["HIV-"]["hubs"].index)
feats.assign(in_Kaforou_27=[g in k27 for g in feats.index], network_hub=[g in hubs for g in feats.index]).round(4)

## 7. External validation on GSE19491
Models are trained on **all HIV− patients in GSE37250** and tested on the active and latent TB patients of GSE19491. The two studies used different array versions (HT-12 V4 vs V3) and labs, so every gene is z-scored within its own dataset (and within each GSE19491 sub-series) before transfer. The probability cut-off is chosen on GSE37250 and applied unchanged.

In [ ]:
ext = pipeline.stage_external(ds, val, cfg, rep)
if ext:
    display(ext["counts"])
    display(ext["table"][["AUC", "AUC_low", "AUC_high", "sensitivity", "specificity"]])
    display(ext["by_site"])

## 8. Answer to the research question
Written automatically from the numbers above, so the text always matches the results.

In [ ]:
answer = pipeline.conclusions(d, m, ext, cfg.target_sensitivity)
rep.sections.insert(0, ("Answer to the research question", answer))
path = rep.write(DATA_NOTE)
display(Markdown(answer))
print("\nFull summary written to", path)

## 9. Limitations and responsible use
* **Who the data represents.** Adults at two Southern African sites (Cape Town, Karonga). Results may not hold in East or West Africa, in children, or in people on long-term ART. Results are reported by site and HIV status above.
* **Validation gap.** GSE19491 contains HIV− patients only (London and Cape Town), so the HIV+ findings have **no independent validation**.
* **Case–control design.** Patients with clear-cut active or latent TB; real clinic populations include harder, intermediate cases, so accuracy here is an upper bound.
* **Secondary use of donated samples.** We use the de-identified public data only for this coursework, make no attempt to re-identify anyone, and cite Kaforou et al. (2013) and Berry et al. (2010).
* **Dual exposure.** A model touching both TB and HIV could reveal HIV status. We share only group-level results, never individual predictions.
* **Not a diagnostic test.** This is research. Any clinical use would need prospective evaluation and confirmation with GeneXpert or culture, with thresholds set and reported separately for HIV+ patients.

In [ ]:
# Reproducibility record
import platform, sklearn, scipy, networkx
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "networkx": networkx.__version__,
            "tbhiv": tbhiv.__version__, "random_state": cfg.random_state}
for pkg in ("GEOparse", "gseapy"):
    try:
        versions[pkg] = __import__(pkg).__version__
    except Exception:
        versions[pkg] = "not installed"
pd.Series(versions).to_csv(cfg.results_dir / "session_info.csv")
versions